## Leetcode 185. Department Top Three Salaries

In [0]:
emp_data = [(1,'Joe',85000,1),(2,'Henry',80000,2),(3,'Sam',60000,2),
            (4,'Max',90000,1),(5,'Janet',69000,1),(6,'Randy',85000,1),(7,'Will',70000,1)]
dept_data = [(1,'IT'),(2,'Sales')]
Employee   = spark.createDataFrame(emp_data,  ['id','name','salary','departmentId'])
Department = spark.createDataFrame(dept_data, ['id','name'])
Employee.createOrReplaceTempView('Employee')
Department.createOrReplaceTempView('Department')


### Spark SQL

In [0]:
spark.sql('''
  SELECT d.name AS Department, e.name AS Employee, e.salary AS Salary
  FROM (SELECT *, DENSE_RANK() OVER (PARTITION BY departmentId ORDER BY salary DESC) rk FROM Employee) e
  JOIN Department d ON e.departmentId = d.id
  WHERE rk <= 3
''').show()

### Spark DataFrame API

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *

In [0]:
spark.conf.set("spark.sql.shuffle.partitions", "1")


emp = Employee.alias('e')
dept = Department.alias('d')

merged=emp.join(dept,emp.departmentId==dept.id)
merged.createOrReplaceTempView('merged')

window= Window.partitionBy('departmentId').orderBy(col('salary').desc())

result = merged.withColumn("rank",dense_rank().over(window))\
    .filter(col("rank")<=3)\
    .select(
        col("d.name").alias("Department"),
        col("e.name").alias("Employee"),
        col("salary").alias("Salary")
    )
result.show()

